# **ML_2. Supervised Learning**

### **1. Ответь на вопросы**

*i. Выведите аналитическое решение задачи регрессии. Используйте векторную форму уравнения.*   

*ii. Какие изменения произойдут в решении при добавлении L1- и L2-регуляризации к функции потерь?*  

*iii. Объясните, почему L1-регуляризация часто используется для отбора признаков. Почему после обучения модели остается много весов, равных 0?*    

*iV. Объясните, как можно использовать те же модели (линейная регрессия, гребневая регрессия и т. д.), но при этом обеспечить возможность учета нелинейных зависимостей*  

### **2. Введение**.  
*Выполните всю предварительную обработку данных из предыдущего урока.*

#### ***i. Импорт библиотек***

In [126]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

#### ***ii. Загрузка train и test датасетов. (read train and test parts)***

In [127]:
data_train = pd.read_json("../datasets/train.json")
print(f"""Количество строк: {data_train.shape[0]}
Количество столбцов: {data_train.shape[1]}""")

Количество строк: 49352
Количество столбцов: 15


In [128]:
data_test = pd.read_json("../datasets/test.json")
print(f"""Количество строк: {data_test.shape[0]}
Количество столбцов: {data_test.shape[1]}""")

Количество строк: 74659
Количество столбцов: 14


In [129]:
percentile_1 = data_train["price"].quantile(0.01)
percentile_99 = data_train["price"].quantile(0.99)

data_train = data_train[
    (data_train["price"] > percentile_1) & (data_train["price"] < percentile_99)
].copy()

price_lowest, price_highest = data_train["price"].min(), data_train["price"].max()


data_train["interest_level"] = data_train["interest_level"].map(
    {"low": 0, "medium": 1, "high": 2}
)
data_train.shape

(48343, 15)

In [130]:
data_test = data_test[
    (data_test["price"] >= price_lowest) & (data_test["price"] <= price_highest)
].copy()

bath_max = data_train["bathrooms"].max()
bed_max = data_train["bedrooms"].max()
data_test = data_test[
    (data_test["bathrooms"] <= bath_max) & (data_test["bedrooms"] <= bed_max)
].copy()

data_test.shape

(73175, 14)

### **3. Введение в анализ данных, часть 2**.  

#### ***i. Давайте создадим дополнительные характеристики для повышения качества модели. Рассмотрим столбец под названием «Features». Он содержит список основных особенностей текущей квартиры.***

In [131]:
cols = ["bathrooms", "bedrooms", "features"]
data_train[cols].head(10)

,bathrooms,bedrooms,features
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di..."
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw..."
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund..."
10,1.5,3,[]
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in..."
16,1.0,3,"[Doorman, Elevator, Loft, Dishwasher, Hardwood..."
18,2.0,3,"[Fireplace, Laundry in Unit, Dishwasher, Hardw..."
19,1.0,0,"[Elevator, Laundry in Building, Dishwasher, Ha..."
23,0.0,1,[Hardwood Floors]
32,3.0,3,"[Cats Allowed, Dogs Allowed]"


#### ***ii. Удалите из столбца неиспользуемые символы ([,], ', ", и пробел).***

*после pd.read_json столбец features это не строка, а список Python. Поэтому для применения строковых методов нужно привести к строке.*

In [132]:
print(type(data_train["features"].iloc[0]))

<class 'list'>


In [133]:
data_train["features"].head(3)

4    [Dining Room, Pre-War, Laundry in Building, Di...
6    [Doorman, Elevator, Laundry in Building, Dishw...
9    [Doorman, Elevator, Laundry in Building, Laund...
Name: features, dtype: object

In [134]:
data_train["features"] = data_train["features"].astype(str)
data_train["features"].iloc[0]

"['Dining Room', 'Pre-War', 'Laundry in Building', 'Dishwasher', 'Hardwood Floors', 'Dogs Allowed', 'Cats Allowed']"

In [135]:
print(type(data_train["features"].iloc[0]))

<class 'str'>


In [136]:
for ch in ["[", "]", "'", '"', " "]:
    data_train["features"] = data_train["features"].str.replace(ch, "", regex=False)

data_train["features"].head(3)

4    DiningRoom,Pre-War,LaundryinBuilding,Dishwashe...
6    Doorman,Elevator,LaundryinBuilding,Dishwasher,...
9    Doorman,Elevator,LaundryinBuilding,LaundryinUn...
Name: features, dtype: str

In [137]:
print(
    "Всего пустых строк  в data_train датасете:", (data_train["features"] == "").sum()
)

Всего пустых строк  в data_train датасете: 3106


#### ***iii. Получите все значения в каждом списке и соберите результат в один большой список для всего набора данных. Для этого можно использовать DataFrame.iterrows().***

In [ ]:
feature_list = []
for _, row in data_train.iterrows():
    if row["features"] != "":
        feature_list.extend(row["features"].split(","))

In [139]:
print("Длина списка:", len(feature_list))

Длина списка: 262377


In [140]:
feature_list[:5]

['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors']

#### ***iv. Сколько уникальных значений содержит список результатов?***

In [141]:
len(set(feature_list))

1529

#### ***v. Давайте познакомимся с новой библиотекой — Collections. С помощью этого пакета вы сможете эффективно получать количественную статистику по вашим данным.***

#### ***vi. Составьте список самых популярных функций из нашего обширного перечня и выберите 20 лучших на данный момент.***

In [142]:
from collections import Counter

In [143]:
feature_counts = Counter(feature_list)

In [158]:
top_20_pairs = feature_counts.most_common(20)
top_20_pairs

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

***Ранее на каггле был другой датасет. Сейчас обновленный. Больше, чем был до этого.***

*пустые строки не учитываю на моменте сбора feature_list

#### ***vii. Если все правильно, вы должны получить следующие значения: 'Elevator', 'CatsAllowed', 'HardwoodFloors', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'.***

данные наименования фич совпадают с полученными (см.выше).

#### ***vi. Теперь создайте 20 новых признаков на основе 20 наиболее часто встречающихся значений: 1, если значение находится в столбце "Feature", в противном случае 0.***
*Now create 20 new features based on the top 20 values: 1 if the value is in the "Feature" column, otherwise 0.*

In [159]:
top_20 = [name for name, _ in top_20_pairs]

In [164]:
splits = data_train["features"].str.split(",")
type(splits)

pandas.Series

In [ ]:
for feature in top_20:
    data_train[feature] = splits.apply(lambda lst: int(feature in lst))

In [167]:
data_train[top_20].head(3)

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0


#### ***ix. Расширьте наш набор признаков, добавив «ванные комнаты», «спальни», и создайте специальную переменную feature_list со всеми названиями признаков. Теперь у нас есть 22 значения. Все модели должны быть обучены на этих 22 признаках.***
Extend our feature set with 'bathrooms', 'bedrooms' and create a special variable feature_list with all feature names. Now we have 22 values. All models should be trained on these 22 features.

In [ ]:
feature_list = top_20 + ["bathrooms", "bedrooms"]
print("Всего признаков:", len(feature_list))

Всего признаков: 22


### **4. Реализация моделей — Линейная регрессия**.  


#### ***i. Реализуйте класс Python для алгоритма линейной регрессии с двумя основными методами — fit и predict. Используйте стохастический градиентный спуск (SGD) для поиска оптимальных весов модели. Для лучшего понимания рекомендуем реализовать отдельные версии алгоритма с аналитическим решением и нестохастическим градиентным спуском.***
implement a Python class for a linear regression algorithm with two basic methods — fit and predict. Use stochastic gradient descent (SGD) to find optimal model weights. For better understanding, we recommend implementing separate versions of the algorithm with the analytical solution and non-stochastic gradient descent under the hood.

#### ***1) Алгоритм с аналитическим решением.***

Аналитическое решение задачи регрессии.  

$$
\boxed{
\hat{\boldsymbol{\beta}}
=
(X^TX)^{-1}X^T\mathbf y
}
$$

#### ***Как решается обратная матрица на практике или почему обычно применяют метод pinv (а не inv) или lstsq, и почему det не диагностика в float64***

В этом проекте X = 22 признака + столбец единиц = 23×23,
матрица полноранговая → XᵀX обратима, np.linalg.inv работает; np.linalg.pinv
здесь страховка (на полноранговой матрице она совпадает с inv). Незаменимой
pinv становится при вырожденной или плохо обусловленной матрице: в ML_1
полином 10 степени от двух базовых признаков (квадратные признаки там остались
только проверкой гипотезы, в модель не пошли) дал 66 столбцов ранга 44 < 66 →
XᵀX сингулярна, np.linalg.inv бросила бы LinAlgError: Singular matrix.

**Почему det не диагностика в float64** (математически det = 0 ⇔ вырожденность):
* масштаб: det(cA) = cⁿ·det(A), при n = 66 отличная матрица даёт det ~1e-46
  или 1e+46 просто из-за масштаба;
* overflow/underflow: det легко уходит в нуль или бесконечность;
* det не показывает степень обусловленности: у катастрофически плохой матрицы
  |det| = 1 (напр., diag(1e10, 1e-10)).

Корректная диагностика — cond = σ_max/σ_min: теряется ~log10(cond) знаков
точности, у float64 запас ~16 знаков, значит cond ~1e16 даёт чистый шум при
det = 1; cond(XᵀX) = cond(X)², поэтому на XᵀX смотрят особенно строго.

**Почему pinv.** Псевдообратная (Мура–Пенроуза) считается через SVD: нулевые
и близкие к нулю сингулярные значения отбрасываются → работает на вырожденных
и плохо обусловленных матрицах, возвращая псевдорешение (least squares)
минимальной нормы. sklearn.LinearRegression решает через SVD
(scipy.linalg.lstsq), поэтому обучилась бы даже на ранге 44.

**Практическое правило:** проверять cond, а не det; решать через pinv/lstsq,
а не через inv.

In [ ]:
# ones — столбец (вектор) из единиц длиной len(data_train):
# по одной единице на каждый объект, чтобы свободный коэффициент (bias)
# стал обычным весом при столбце-единице
# ones = np.ones((len(data_train), 1))
# ones = np.ones((len(data_train), 1))
# ones

array([[1.],
       [1.],
       [1.],
       ...,
       [1.],
       [1.],
       [1.]], shape=(48343, 1))

In [ ]:
# np.hstack([data_train, ones])#

array([[1.0, 1, '8579a0b0d54db803821a35a4a615e97a', ..., 0,
        2556.2328579138893, 1.0],
       [1.0, 2, 'b8e75fc949a6cd8225b455648a951712', ..., 0,
        3537.6649344373436, 1.0],
       [1.0, 2, 'cd759a988b8f23924b5a2058d5ab2b49', ..., 0,
        4154.965711762599, 1.0],
       ...,
       [1.0, 1, '67997a128056ee1ed7d046bbb856e3c7', ..., 0,
        3072.8187639867206, 1.0],
       [1.0, 2, '3c0574a740154806c18bdf1fddd3d966', ..., 0,
        3418.063110671622, 1.0],
       [1.0, 3, 'd89f514c3ed0abaae52cba7017ac0701', ..., 0,
        3498.4859054416397, 1.0]], shape=(48343, 37), dtype=object)

In [ ]:
class LinearRegressionAnalytic:
    def fit(self, X, y):
        X = np.asarray(
            X, dtype=float
        )  # np.array() cсоздает новую копию, а np.asarray копирует только когда это действительно нужно: список, DataFrame, другой dtype. Если вход уже float-массив, asarray вернет тот же объект без копирования.
        y = np.asarray(y, dtype=float)
        X_b = np.hstack([X, np.ones((len(X), 1))])  # столбец единиц для bias
        self.w = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_b = np.hstack([X, np.ones((len(X), 1))])
        return X_b @ self.w

In [ ]:
X_train = data_train[feature_list].to_numpy()
y_train = data_train["price"].to_numpy()

lr_analytic = LinearRegressionAnalytic().fit(X_train, y_train)
y_pred_analytic = lr_analytic.predict(X_train)

print("Размер вектора весов:", lr_analytic.w.shape)  # (23,) = 22 признака + bias
print("MAE train: ", round(mean_absolute_error(y_train, y_pred_analytic), 2))
print("RMSE train:", round(mean_squared_error(y_train, y_pred_analytic) ** 0.5, 2))

data_train["pred_lr_analytic"] = y_pred_analytic

Размер вектора весов: (23,)
MAE train:  708.74
RMSE train: 1027.26


In [ ]:
https://chat.qwen.ai/c/14bff9f6-6c54-4b90-b5d8-8363c1f44c33

#### ***ii. Что такое детерминированная модель? Сделайте SGD детерминированным.***
What is determenistic model? Make SGD determenistic.

#### ***iii. Определите коэффициент R² и реализуйте функцию для его вычисления.***
Define the R squared (R2) coefficient and implement a function to calculate it.

#### ***iv. С помощью вашего алгоритма сделайте прогнозы и оцените модель, используя метрики MAE, RMSE и R2.***
Make predictions with your algorithm and estimate the model with MAE, RMSE and R2 metrics.

#### ***v. Инициализируйте функцию LinearRegression() из sklearn.linear_model, обучите модель и выполните прогнозирование для обучающей и тестовой частей, как в предыдущем уроке.***
Initialize LinearRegression() from sklearn.linear_model, fit the model, and predict the training and test parts as in the previous lesson.

#### ***vi. Сравните показатели качества и убедитесь, что разница невелика (между вашей реализацией и sklearn).***
Compare the quality metrics and make sure the difference is small (between your implementations and sklearn).

#### ***vii. Сохраните метрики, как и в предыдущем уроке, в таблице со столбцами model, train, test для таблицы MAE, таблицы RMSE и коэффициента R2.***
Store the metrics as in the previous lesson in a table with columns model, train, test for MAE table, RMSE table, and R2 coefficient.